# Създаване на приложения за генериране на изображения (Azure OpenAI)

LLM моделите предлагат повече от текстово генериране. Можете също да генерирате изображения от текстови описания. Изображенията като модалност са полезни в Медицински технологии, архитектура, туризъм, разработка на игри, маркетинг и други. В този урок работим с днешните **GPT Image** модели в Microsoft Foundry.

## Учебни цели

В края на този урок ще можете да:

- Обясните какво е генериране на изображения и къде е полезно.
- Разберете семейството модели `gpt-image` и как се различава от наследствените модели DALL·E.
- Създадете приложение за генериране на изображения и редактирате изображения.

## Какво е генериране на изображения?

Моделите за генериране на изображения създават изображения от текстово подтикване. Модерните модели като `gpt-image` изучават връзката между текст и изображения по време на обучение, след което итеративно превръщат случаен шум в изображение, което съответства на вашето подтикване.

Два познати семейства модели за изображения са:

- **`gpt-image` (OpenAI)** — актуалното поколение, използвано в този урок. Поддържа генериране на изображения от текст и редактиране на изображения (inpainting с маска).
- **Midjourney** — популярен модел на трета страна с отделна услуга и работен процес базиран на Discord.

> По-старите модели за изображения на OpenAI — **DALL·E 2** и **DALL·E 3** — са наследствени. DALL·E 3 вече не е наличен за нови внедрявания, а функции като `create_variation` съществуваха само в DALL·E 2. Използвайте моделите `gpt-image` за нови приложения.

В Microsoft Foundry, **`gpt-image-2`** е най-новият и най-способният модел за изображения и е препоръчителният по подразбиране. `gpt-image-1.5` и `gpt-image-1-mini` също са общодостъпни.

> **Важно:** Моделите `gpt-image` връщат генерираното изображение като **base64** (`b64_json`), а не като URL адрес. Вашият код декодира base64 низа в байтове и го записва — няма URL адрес на изображение за сваляне.


## Създаване на вашето първо приложение за генериране на изображения

Какво е необходимо, за да създадете приложение за генериране на изображения? Трябва ви следните библиотеки:

- **python-dotenv**, силно се препоръчва да използвате тази библиотека, за да съхранявате тайните си в *.env* файл, отделен от кода.
- **openai**, тази библиотека ще използвате за взаимодействие с OpenAI API.
- **pillow**, за работа с изображения в Python.

Ако все още не сте го направили, следвайте инструкциите на страницата [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst), за да създадете ресурс и модел в Microsoft Foundry. Изберете **gpt-image-2** като модел (най-новият модел за изображения на Azure OpenAI; DALL·E е остарял).

1. Създайте файл *.env* със следното съдържание:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Намерете тази информация в портала Microsoft Foundry за вашия ресурс в секцията "Deployments".



1. Съберете горните библиотеки във файл, наречен *requirements.txt*, по следния начин:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. След това, създайте виртуална среда и инсталирайте библиотеките:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> За Windows използвайте следните команди, за да създадете и активирате виртуалната си среда:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Добавете следния код във файл, наречен *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # конфигурирайте клиента Azure OpenAI (Microsoft Foundry).
    # Моделите за изображения изискват актуална версия на API — проверете документацията на Foundry за версията, която вашият модел изисква.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Създайте изображение с помощта на API за генериране на изображения
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Въведете текста на вашия подканващ низ тук
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # напр. "gpt-image-2"
        )
        # Задайте директорията за съхраняване на изображението
        image_dir = os.path.join(os.curdir, 'images')

        # Ако директорията не съществува, създайте я
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Инициализирайте пътя до изображението (забележете, че типът на файла трябва да е png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # моделите gpt-image връщат изображението като base64 (b64_json), а не URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Показване на изображението в стандартния прегледач за изображения
        image = Image.open(image_path)
        image.show()

    # обработка на изключения
    except BadRequestError as err:
        print(err)

    ```

Нека обясним този код:

- Първо, импортираме нужните библиотеки, включително библиотеката OpenAI, библиотеката dotenv, модула base64 и библиотеката Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- След това зареждаме променливите на средата от файла *.env*.

    ```python
    # импорт на dotenv
    dotenv.load_dotenv()
    ```

- След това конфигурираме клиента за Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- След това генерираме изображението:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Въведете текста на вашия подсказване тук
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` моделите връщат изображението като **base64** низ в `data[0].b64_json`. Декодираме го до байтове и го записваме във файл — няма URL за сваляне.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Накрая, отваряме изображението и използваме стандартния визуализатор за изображения, за да го покажем:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Повече подробности за генериране на изображението

Нека разгледаме параметрите на `images.generate`:

- **prompt**, е текстовата подсказка, използвана за генериране на изображението. Тук е "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, е размерът на генерираното изображение (например `1024x1024`, `1536x1024`, `1024x1536` или `"auto"`).
- **n**, е броят на генерираните изображения. Тук генерираме едно.
- **model**, е името на вашето разгръщане на модел за изображения (например `gpt-image-2`).

> Моделите за изображения не приемат параметър `temperature` — това е контрол за генериране на текст. За повече разнообразие, извикайте API-то отново; за по-малко разнообразие, направете подсказката си по-конкретна.

## Допълнителни възможности при генериране на изображения

Видяхте как да генерирате изображение с няколко реда Python. `gpt-image` моделите могат също да **редактират** съществуващо изображение. Като предоставите изображение, опционална **маска** (която маркира зоната за промяна) и подсказка, можете да промените част от изображението — например да добавите шапка на зайчето ни.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# редакциите също се връщат като base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Основното изображение съдържа само зайчето; крайното изображение има шапка на зайчето.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
